# Analysis: B1 Persistent buffer_partial_lh

**Comparison of:** `2026_03_16_bl_estimation_opt` (T2 baseline) vs `2026_03_16_bl_estimation_buffer` (B1: persistent buffer)  
**Dataset:** 100 taxa, 1M sites, 10 different trees, 10 runs each  
**Backend:** OPENACC (GPU V100)  
**Models:** DNA/GTR unrooted, AA/LG unrooted  

**Change:** Replace per-call `exit data delete` + `enter data copyin` with persistent GPU allocation (`enter data copyin` once, `update device` thereafter). Eliminates GPU malloc/free cycle per likelihood call.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cwd = os.getcwd()
t2_path = "/Users/u7826985/Projects/Nvidia/results/2026_03_16_bl_estimation_opt"
b1_path = "/Users/u7826985/Projects/Nvidia/results/2026_03_16_bl_estimation_buffer"

# 1. Parse all log files

In [ ]:
alignment_pattern = re.compile(r"Alignment has (\d+) sequences with (\d+) columns, (\d+) distinct patterns")
initial_ll_pattern = re.compile(r"Initial log-likelihood:\s+([-0-9.]+)")
optimal_ll_pattern = re.compile(r"Optimal log-likelihood:\s+([-0-9.]+)")
params_opt_pattern = re.compile(r"Parameters optimization took (\d+) rounds? \(([0-9.]+) sec\)")
wallclock_pattern = re.compile(r"Total wall-clock time used:\s+([0-9.]+)\s+sec")

In [ ]:
def parse_all_logs(results_path, source_label="baseline"):
    """Parse all IQ-TREE log files and extract timing/likelihood data."""
    rows = []
    skipped = 0

    for data_type in ['AA', 'DNA']:
        data_dir = os.path.join(results_path, data_type)
        if not os.path.exists(data_dir):
            continue

        for tree_type in ['rooted', 'unrooted']:
            tree_type_dir = os.path.join(data_dir, tree_type)
            if not os.path.exists(tree_type_dir):
                continue

            for model in sorted(os.listdir(tree_type_dir)):
                model_dir = os.path.join(tree_type_dir, model)
                if not os.path.isdir(model_dir):
                    continue

                for tree_folder in sorted(os.listdir(model_dir)):
                    tree_dir = os.path.join(model_dir, tree_folder)
                    if not os.path.isdir(tree_dir):
                        continue

                    for fname in os.listdir(tree_dir):
                        if not fname.endswith('.log'):
                            continue

                        filepath = os.path.join(tree_dir, fname)
                        with open(filepath, 'r') as f:
                            content = f.read()

                        # Determine backend from filename
                        if 'OPENACC' in fname:
                            backend = 'GPU_V100'
                        elif 'OMP_48' in fname:
                            backend = 'CPU_48cores'
                        elif 'OMP_10' in fname:
                            backend = 'CPU_10cores'
                        elif 'VANILA' in fname:
                            backend = 'CPU_1core'
                        else:
                            continue

                        run_matches = re.findall(r'_run(\d+)_', fname)
                        run_number = int(run_matches[-1]) if run_matches else None

                        aln_match = alignment_pattern.search(content)
                        init_ll_match = initial_ll_pattern.search(content)
                        opt_ll_match = optimal_ll_pattern.search(content)
                        opt_match = params_opt_pattern.search(content)
                        wc_match = wallclock_pattern.search(content)

                        if aln_match and init_ll_match and opt_match and wc_match:
                            rows.append({
                                'source': source_label,
                                'data_type': data_type,
                                'tree_type': tree_type,
                                'model': model,
                                'treefile': tree_folder,
                                'backend': backend,
                                'run': run_number,
                                'taxa': int(aln_match.group(1)),
                                'sites': int(aln_match.group(2)),
                                'patterns': int(aln_match.group(3)),
                                'initial_likelihood': float(init_ll_match.group(1)),
                                'optimal_likelihood': float(opt_ll_match.group(1)) if opt_ll_match else float(init_ll_match.group(1)),
                                'opt_rounds': int(opt_match.group(1)),
                                'opt_time': float(opt_match.group(2)),
                                'wallclock_time': float(wc_match.group(1)),
                            })
                        else:
                            skipped += 1

    df = pd.DataFrame(rows)
    print(f"[{source_label}] Parsed {len(df)} log files ({skipped} skipped)")
    return df

df_t2 = parse_all_logs(t2_path, "T2 (baseline)")
df_b1 = parse_all_logs(b1_path, "B1 (buffer)")

# 2. Data overview

In [ ]:
print("=== T2 (baseline) data breakdown ===")
print(df_t2.groupby(['data_type','tree_type','model','backend']).size().reset_index(name='count'))
print(f"\n=== B1 (buffer) data breakdown ===")
print(df_b1.groupby(['data_type','tree_type','model','backend']).size().reset_index(name='count'))

# Combine
df_all = pd.concat([df_t2, df_b1], ignore_index=True)
df_all['combo'] = df_all['data_type'] + '\n' + df_all['model'] + '\n' + df_all['tree_type']

# 3. Statistical comparison

In [ ]:
print("=" * 80)
print("COMPARISON: T2 (baseline) vs B1 (persistent buffer_partial_lh)")
print("  T2 = exit data delete + enter data copyin every call")
print("  B1 = enter data copyin once, update device thereafter")
print("=" * 80)

combos = df_b1[['data_type','tree_type','model','backend']].drop_duplicates()

for _, combo in combos.iterrows():
    dt, tt, model, backend = combo['data_type'], combo['tree_type'], combo['model'], combo['backend']
    t2_sub = df_t2[(df_t2['data_type']==dt) & (df_t2['tree_type']==tt) & 
                   (df_t2['model']==model) & (df_t2['backend']==backend)]
    b1_sub = df_b1[(df_b1['data_type']==dt) & (df_b1['tree_type']==tt) & 
                   (df_b1['model']==model) & (df_b1['backend']==backend)]
    
    print(f"\n--- {dt} | {tt} | {model} | {backend} ---")
    print(f"  T2:  n={len(t2_sub)}, wallclock median={t2_sub['wallclock_time'].median():.3f}s, "
          f"mean={t2_sub['wallclock_time'].mean():.3f}s, std={t2_sub['wallclock_time'].std():.3f}s")
    print(f"  B1:  n={len(b1_sub)}, wallclock median={b1_sub['wallclock_time'].median():.3f}s, "
          f"mean={b1_sub['wallclock_time'].mean():.3f}s, std={b1_sub['wallclock_time'].std():.3f}s")
    if t2_sub['wallclock_time'].median() > 0:
        pct = (b1_sub['wallclock_time'].median() - t2_sub['wallclock_time'].median()) / t2_sub['wallclock_time'].median() * 100
        print(f"  Change (median): {pct:+.2f}%")
    
    # opt_time comparison
    if len(t2_sub) > 0 and 'opt_time' in t2_sub.columns:
        print(f"  T2 opt_time: median={t2_sub['opt_time'].median():.3f}s")
        print(f"  B1 opt_time: median={b1_sub['opt_time'].median():.3f}s")
        pct2 = (b1_sub['opt_time'].median() - t2_sub['opt_time'].median()) / t2_sub['opt_time'].median() * 100
        print(f"  Opt_time change (median): {pct2:+.2f}%")
    
    # LL comparison
    print(f"  T2 optimal_LL: {t2_sub['optimal_likelihood'].unique()}")
    print(f"  B1 optimal_LL: {b1_sub['optimal_likelihood'].unique()}")
    t2_lls = set(t2_sub['optimal_likelihood'].unique())
    b1_lls = set(b1_sub['optimal_likelihood'].unique())
    if t2_lls == b1_lls:
        print(f"  LL match: YES (exact match)")
    else:
        print(f"  LL match: NO — INVESTIGATE!")

# 4. Plot setup

In [ ]:
SOURCE_PALETTE = {'T2 (baseline)': '#1565C0', 'B1 (buffer)': '#E65100'}

sns.set_style("whitegrid")
sns.set_context("talk")
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 13,
    'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'legend.fontsize': 11, 'figure.titlesize': 16,
    'axes.linewidth': 1.2, 'lines.linewidth': 2,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.15,
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'font.family': 'sans-serif',
})

# 5. Wall-clock time per tree

In [ ]:
combos = df_b1[['data_type','tree_type','model']].drop_duplicates()
n_combos = len(combos)
fig, axes = plt.subplots(1, n_combos, figsize=(11 * n_combos, 7))
if n_combos == 1:
    axes = [axes]

for idx, (_, combo) in enumerate(combos.iterrows()):
    dt, tt, model = combo['data_type'], combo['tree_type'], combo['model']
    subset = df_all[(df_all['data_type']==dt) & (df_all['tree_type']==tt) & (df_all['model']==model)]
    ax = axes[idx]
    sns.boxplot(data=subset, x='treefile', y='wallclock_time', hue='source',
                hue_order=['T2 (baseline)', 'B1 (buffer)'], palette=SOURCE_PALETTE,
                linewidth=1.2, ax=ax)
    ax.set_title(f'{dt} {model} {tt}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Tree')
    ax.set_ylabel('Wall-clock Time (s)')
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    if idx > 0:
        ax.get_legend().remove()
    else:
        ax.legend(title='Version', fontsize=10)

plt.suptitle('T2 vs B1: Wall-clock Time per Tree (GPU V100)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{cwd}/comparison_wallclock_per_tree.png", bbox_inches='tight', dpi=300)
plt.show()

# 6. Aggregated comparison (wallclock + opt_time)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, metric, label in [(ax1, 'wallclock_time', 'Wall-clock Time (s)'), 
                           (ax2, 'opt_time', 'Optimization Time (s)')]:
    sns.boxplot(data=df_all, x='combo', y=metric, hue='source',
                hue_order=['T2 (baseline)', 'B1 (buffer)'], palette=SOURCE_PALETTE,
                linewidth=1.2, ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel(label, fontsize=12)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(title='Version', fontsize=10)
    ax.tick_params(axis='x', labelsize=10)

plt.suptitle('T2 vs B1: Aggregated Comparison (GPU V100)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{cwd}/comparison_aggregated.png", bbox_inches='tight', dpi=300)
plt.show()

# 7. Per-tree median comparison table

In [ ]:
combos = df_b1[['data_type','tree_type','model','backend']].drop_duplicates()

for _, combo in combos.iterrows():
    dt, tt, model, backend = combo['data_type'], combo['tree_type'], combo['model'], combo['backend']
    print(f"\n{'='*70}")
    print(f"  {dt}/{model} ({tt}) — {backend}")
    print(f"{'='*70}")
    print(f"{'Tree':<10} {'T2 median':>12} {'B1 median':>12} {'Diff (s)':>10} {'Change':>10} {'LL match':>10}")
    print(f"{'-'*10} {'-'*12} {'-'*12} {'-'*10} {'-'*10} {'-'*10}")

    all_t2 = []
    all_b1 = []

    for tree in sorted(df_t2[(df_t2['data_type']==dt) & (df_t2['model']==model)]['treefile'].unique()):
        t2_times = df_t2[(df_t2['data_type']==dt) & (df_t2['tree_type']==tt) & 
                         (df_t2['model']==model) & (df_t2['treefile']==tree)]['wallclock_time']
        b1_times = df_b1[(df_b1['data_type']==dt) & (df_b1['tree_type']==tt) & 
                         (df_b1['model']==model) & (df_b1['treefile']==tree)]['wallclock_time']
        t2_lls = df_t2[(df_t2['data_type']==dt) & (df_t2['tree_type']==tt) & 
                       (df_t2['model']==model) & (df_t2['treefile']==tree)]['optimal_likelihood']
        b1_lls = df_b1[(df_b1['data_type']==dt) & (df_b1['tree_type']==tt) & 
                       (df_b1['model']==model) & (df_b1['treefile']==tree)]['optimal_likelihood']

        if len(t2_times) == 0 or len(b1_times) == 0:
            continue

        t2_med = t2_times.median()
        b1_med = b1_times.median()
        diff = b1_med - t2_med
        pct = (diff / t2_med) * 100
        ll_match = "YES" if set(t2_lls.unique()) == set(b1_lls.unique()) else "NO"

        all_t2.append(t2_med)
        all_b1.append(b1_med)

        print(f"{tree:<10} {t2_med:>12.3f} {b1_med:>12.3f} {diff:>+10.3f} {pct:>+9.1f}% {ll_match:>10}")

    if all_t2 and all_b1:
        avg_t2 = np.mean(all_t2)
        avg_b1 = np.mean(all_b1)
        med_t2 = np.median(all_t2)
        med_b1 = np.median(all_b1)
        print(f"{'-'*10} {'-'*12} {'-'*12} {'-'*10} {'-'*10} {'-'*10}")
        print(f"{'MEAN':<10} {avg_t2:>12.3f} {avg_b1:>12.3f} {avg_b1-avg_t2:>+10.3f} {(avg_b1-avg_t2)/avg_t2*100:>+9.1f}%")
        print(f"{'MEDIAN':<10} {med_t2:>12.3f} {med_b1:>12.3f} {med_b1-med_t2:>+10.3f} {(med_b1-med_t2)/med_t2*100:>+9.1f}%")

# 8. Per-run speedup distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ratios_list = []
combos = df_b1[['data_type','tree_type','model','backend']].drop_duplicates()

for _, combo in combos.iterrows():
    dt, tt, model, backend = combo['data_type'], combo['tree_type'], combo['model'], combo['backend']
    for tree in [f'tree_{i}' for i in range(1, 11)]:
        for run in range(1, 11):
            t2_val = df_t2[(df_t2['data_type']==dt) & (df_t2['tree_type']==tt) & 
                           (df_t2['model']==model) & (df_t2['backend']==backend) &
                           (df_t2['treefile']==tree) & (df_t2['run']==run)]['wallclock_time']
            b1_val = df_b1[(df_b1['data_type']==dt) & (df_b1['tree_type']==tt) & 
                           (df_b1['model']==model) & (df_b1['backend']==backend) &
                           (df_b1['treefile']==tree) & (df_b1['run']==run)]['wallclock_time']
            if len(t2_val) == 1 and len(b1_val) == 1:
                ratio = t2_val.values[0] / b1_val.values[0]
                ratios_list.append({
                    'combo': f'{dt}\n{model}\n{tt}',
                    'speedup': ratio
                })

df_ratios = pd.DataFrame(ratios_list)
if not df_ratios.empty:
    sns.boxplot(data=df_ratios, x='combo', y='speedup', linewidth=1.2,
                color='#E65100', ax=ax)
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='No change (1.0x)')
    ax.set_xlabel('')
    ax.set_ylabel('Speedup (T2 / B1)', fontsize=12)
    ax.set_title('Per-Run Speedup: T2 \u2192 B1 (persistent buffer)', fontsize=14, fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{cwd}/comparison_speedup_distribution.png", bbox_inches='tight', dpi=300)
    plt.show()

# 9. Correctness: Log-likelihood comparison

In [ ]:
print("=" * 70)
print("Correctness: Log-likelihood comparison (T2 vs B1)")
print("=" * 70)

all_match = True
combos = df_b1[['data_type','tree_type','model']].drop_duplicates()

for _, combo in combos.iterrows():
    dt, tt, model = combo['data_type'], combo['tree_type'], combo['model']
    for tree in sorted(df_b1[(df_b1['data_type']==dt) & (df_b1['model']==model)]['treefile'].unique()):
        t2_lls = set(df_t2[(df_t2['data_type']==dt) & (df_t2['tree_type']==tt) & 
                           (df_t2['model']==model) & (df_t2['treefile']==tree)]['optimal_likelihood'].unique())
        b1_lls = set(df_b1[(df_b1['data_type']==dt) & (df_b1['tree_type']==tt) & 
                           (df_b1['model']==model) & (df_b1['treefile']==tree)]['optimal_likelihood'].unique())
        if t2_lls != b1_lls:
            print(f"  MISMATCH {dt}/{model}/{tt}/{tree}: T2={t2_lls} vs B1={b1_lls}")
            all_match = False

if all_match:
    print("  All log-likelihoods match exactly (0.0 diff) across all trees and runs")
    print("  Correctness: VERIFIED")

# 10. Summary

In [ ]:
print("=" * 70)
print("SUMMARY: B1 Persistent buffer_partial_lh")
print("=" * 70)
print()
print("Change: Replace per-call GPU alloc/dealloc (exit data delete +")
print("        enter data copyin) with persistent allocation (enter data")
print("        copyin once, update device on subsequent calls).")
print()

combos = df_b1[['data_type','tree_type','model','backend']].drop_duplicates()
print(f"{'Model':<15} {'T2 median':>12} {'B1 median':>12} {'Change':>10}")
print(f"{'-'*15} {'-'*12} {'-'*12} {'-'*10}")

for _, combo in combos.iterrows():
    dt, tt, model, backend = combo['data_type'], combo['tree_type'], combo['model'], combo['backend']
    t2_med = df_t2[(df_t2['data_type']==dt) & (df_t2['tree_type']==tt) & 
                   (df_t2['model']==model) & (df_t2['backend']==backend)]['wallclock_time'].median()
    b1_med = df_b1[(df_b1['data_type']==dt) & (df_b1['tree_type']==tt) & 
                   (df_b1['model']==model) & (df_b1['backend']==backend)]['wallclock_time'].median()
    pct = (b1_med - t2_med) / t2_med * 100
    label = f"{dt}/{model}"
    print(f"{label:<15} {t2_med:>12.3f} {b1_med:>12.3f} {pct:>+9.1f}%")

print()
print("Correctness: All log-likelihoods match exactly")
print("Verdict: Neutral — GPU malloc/free overhead was not a bottleneck")
print("Keep: Yes (cleaner code, matches derivative function pattern)")